## Fast Zero-Shot

In [1]:
import subprocess

namespace = "dsc-capstone-26-27"
job_name = "esther-babylm-fast-eval"

result = subprocess.run(
    ["kubectl", "logs", f"job/{job_name}", "-n", namespace],
    capture_output=True,
    text=True,
    check=True
)

logs = result.stdout

print("Retrieved evaluation logs from Nautilus!")
print(logs[-2000:])

Retrieved evaluation logs from Nautilus!
6 [00:13<00:04, 93.99it/s]
100%|██████████| 1726/1726 [00:17<00:00, 97.26it/s]
EYE TRACKING SCORE: 0.65
SELF-PACED READING SCORE: 0.06
Fast evaluation completed!



In [2]:

import re
import pandas as pd
from IPython.display import display

# Extract the average accuracy reported by each benchmark
accuracy_scores = re.findall(
    r"### AVERAGE ACCURACY\s*\n([\d.]+)",
    logs
)

# Extract reading benchmark scores
eye_tracking = re.search(
    r"EYE TRACKING SCORE:\s*([\d.]+)", logs
)
self_paced = re.search(
    r"SELF-PACED READING SCORE:\s*([\d.]+)", logs
)

benchmarks = [
    "BLiMP",
    "BLiMP Supplement",
    "EWoK",
    "Entity Tracking"
]

results = pd.DataFrame({
    "Benchmark": benchmarks + [
        "Eye Tracking",
        "Self-Paced Reading"
    ],
    "Score": [
        *[float(score) for score in accuracy_scores],
        float(eye_tracking.group(1)),
        float(self_paced.group(1))
    ],
    "Metric": ["Accuracy (%)"] * 4 + ["Reading score"] * 2
})

display(results)


,Benchmark,Score,Metric
0,BLiMP,63.42,Accuracy (%)
1,BLiMP Supplement,54.80,Accuracy (%)
2,EWoK,48.18,Accuracy (%)
3,Entity Tracking,17.72,Accuracy (%)
4,Eye Tracking,0.65,Reading score
5,Self-Paced Reading,0.06,Reading score


## Full Zero Shot

In [3]:
import subprocess

namespace = "dsc-capstone-26-27"
job_name = "esther-babylm-full-eval"

result = subprocess.run(
    ["kubectl", "logs", f"job/{job_name}", "-n", namespace],
    capture_output=True,
    text=True,
    check=True
)

full_logs = result.stdout

print("Retrieved full evaluation logs from Nautilus!")
print(full_logs[-2000:])

Retrieved full evaluation logs from Nautilus!
0:03, 258.97it/s]
100%|██████████| 1726/1726 [00:07<00:00, 245.47it/s]
EYE TRACKING SCORE: 0.65
SELF-PACED READING SCORE: 0.06
Fast evaluation completed!



In [4]:
import re
import pandas as pd
from IPython.display import display

# Extract overall accuracies from full evaluation logs
accuracy_scores = re.findall(
    r"### AVERAGE ACCURACY\s*\n([\d.]+)",
    full_logs
)

# Extract reading benchmark scores
eye_tracking = re.search(
    r"EYE TRACKING SCORE:\s*([\d.]+)", full_logs
)
self_paced = re.search(
    r"SELF-PACED READING SCORE:\s*([\d.]+)", full_logs
)

benchmarks = [
    "BLiMP",
    "BLiMP Supplement",
    "EWoK",
    "Entity Tracking",
    "COMPS"
]

assert len(accuracy_scores) == 5, (
    f"Expected 5 accuracy scores, found {len(accuracy_scores)}"
)

full_results = pd.DataFrame({
    "Benchmark": benchmarks + [
        "Eye Tracking",
        "Self-Paced Reading"
    ],
    "Score": (
        [float(score) for score in accuracy_scores]
        + [float(eye_tracking.group(1)), float(self_paced.group(1))]
    ),
    "Metric": ["Accuracy (%)"] * 5 + ["Reading score"] * 2
})

display(full_results)

,Benchmark,Score,Metric
0,BLiMP,63.75,Accuracy (%)
1,BLiMP Supplement,53.79,Accuracy (%)
2,EWoK,49.07,Accuracy (%)
3,Entity Tracking,18.08,Accuracy (%)
4,COMPS,50.44,Accuracy (%)
5,Eye Tracking,0.65,Reading score
6,Self-Paced Reading,0.06,Reading score


In [5]:
import re
import pandas as pd
from IPython.display import display

# Split the logs into individual benchmark sections
sections = re.split(r"### AVERAGE ACCURACY\s*\n[\d.]+", full_logs)

benchmarks = [
    "BLiMP",
    "BLiMP Supplement",
    "EWoK",
    "Entity Tracking",
    "COMPS"
]

# Extract each benchmark's individual task accuracies
task_results = {}

for benchmark, section in zip(benchmarks, sections):
    match = re.search(
        r"### UID ACCURACY\s*\n(.*?)(?=\n### |\Z)",
        section,
        re.DOTALL
    )

    if match:
        tasks = re.findall(
            r"^(.+?):\s*([\d.]+)$",
            match.group(1),
            re.MULTILINE
        )

        task_results[benchmark] = pd.DataFrame(
            tasks,
            columns=["Task", "Accuracy (%)"]
        )

        task_results[benchmark]["Accuracy (%)"] = (
            task_results[benchmark]["Accuracy (%)"].astype(float)
        )

# Display each benchmark's task accuracies
for benchmark, df in task_results.items():
    print(f"\n{benchmark} — Individual Task Accuracies")
    display(df)


BLiMP — Individual Task Accuracies


,Task,Accuracy (%)
0,coordinate_structure_constraint_object_extraction,38.88
1,left_branch_island_echo_question,45.93
2,wh_island,46.35
3,anaphor_number_agreement,90.23
4,irregular_plural_subject_verb_agreement_1,75.25
...,...,...
62,determiner_noun_agreement_with_adj_irregular_2,78.10
63,principle_A_c_command,54.76
64,ellipsis_n_bar_1,37.16
65,tough_vs_raising_2,79.02



BLiMP Supplement — Individual Task Accuracies


,Task,Accuracy (%)
0,qa_congruence_tricky,29.09
1,qa_congruence_easy,46.88
2,turn_taking,54.64
3,subject_aux_inversion,87.25
4,hypernym,51.07



EWoK — Individual Task Accuracies


,Task,Accuracy (%)
0,quantitative-properties,51.59
1,physical-dynamics,43.33
2,material-properties,52.35
3,social-properties,44.21
4,agent-properties,50.14
5,physical-interactions,46.94
6,social-relations,50.19
7,material-dynamics,46.49
8,social-interactions,52.38
9,physical-relations,50.49



Entity Tracking — Individual Task Accuracies


,Task,Accuracy (%)
0,ambiref_0_ops,17.13
1,ambiref_1_ops,14.72
2,ambiref_2_ops,15.50
3,ambiref_3_ops,18.58
4,ambiref_4_ops,14.52
5,ambiref_5_ops,22.76
6,move_contents_0_ops,18.02
7,move_contents_1_ops,18.76
8,move_contents_2_ops,16.29
9,move_contents_3_ops,16.01



COMPS — Individual Task Accuracies


,Task,Accuracy (%)
0,wugs,49.35
1,base,53.07
2,wugs_dist_before,48.96
3,wugs_dist_in_between,50.38


## Full Fine-Tuning Evaluation Results

In [13]:
import subprocess
import pandas as pd
import re

command = [
    "kubectl", "exec",
    "-n", "dsc-capstone-26-27",
    "esther-storage", "--",
    "find",
    "/shared/esther/babylm-eval/strict/results/neefqhar",
    "-path", "*/finetune/*/results.txt",
    "-exec", "sh", "-c",
    'echo "=== $1 ==="; cat "$1"; echo',
    "_", "{}", ";"
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Error:", result.stderr)

records = []
current_task = None

for line in result.stdout.splitlines():
    if line.startswith("==="):
        path = line.strip("= ").strip()
        current_task = path.split("/")[-2].upper()
        records.append({"Task": current_task})
    elif ":" in line and current_task:
        metric, value = line.split(":", 1)
        if metric.strip() in {"accuracy", "f1", "mcc"}:
            records[-1][metric.strip().upper()] = float(value.strip())

df_finetuning = pd.DataFrame(records)

task_order = ["BOOLQ", "MULTIRC", "RTE", "WSC", "MRPC", "QQP", "MNLI"]

df_finetuning["Task"] = pd.Categorical(
    df_finetuning["Task"],
    categories=task_order,
    ordered=True
)

df_finetuning = df_finetuning.sort_values("Task").reset_index(drop=True)

display(
    df_finetuning.style.format({
        "ACCURACY": "{:.2%}",
        "F1": "{:.3f}",
        "MCC": "{:.3f}"
    }, na_rep="—")
)

Return code: 0
Error: 


,Task,ACCURACY,F1,MCC
0,BOOLQ,66.73%,0.760,0.236
1,MULTIRC,60.15%,0.474,0.164
2,RTE,58.99%,0.544,0.172
3,WSC,63.46%,0.240,0.144
4,MRPC,70.10%,0.820,0.207
5,QQP,66.77%,0.611,0.338
6,MNLI,48.06%,—,—
